# KPIs Ventas — Coach Comercial

Flujo de datos:
1. **forecast** → budget/forecast por línea (filtro: `Zona de ventas` + `Agrupacion de Mes y Año`)
2. **cabecera** → facturas del vendedor/mes (`Zona de ventas` + `Fecha de factura`)
3. **posición** → litros reales por línea (join por `Factura`)

Ejecutar la celda final e ingresar vendedor y período por consola.

In [9]:
from datetime import date, datetime
from pathlib import Path

import pandas as pd

# ── Configuración ──────────────────────────────────────────────────────────────
DATASET = Path("dataset")

ANO_ACTUAL = date.today().year
MES_ACTUAL = date.today().month

VENDEDORES = {
    "38": {"nombre": "Rivero, Tomás",        "zona": "OFF - GBA Norte"},
    "43": {"nombre": "Montes de Oca, Jorge", "zona": "KAM-Supermercados 2"},
    "72": {"nombre": "Gil, Germán",           "zona": "ON-KAM"},
    "77": {"nombre": "Bozzini, Nicolas",      "zona": "ON - KAM"},
}

LINEAS_PRINCIPALES = {
    "Fernet Branca":  ["Fernet Branca", "Brancamenta", "Fernet Vittone"],
    "Sernova":        ["Sernova Vodka"],
    "Carpano":        ["Carpano"],
    "Antica Formula": ["Antica Fórmula", "Antica Formula"],
}


# ── Helpers ────────────────────────────────────────────────────────────────────
def parse_num_ar(value) -> float:
    """Número con formato argentino: punto miles, coma decimal."""
    if pd.isna(value):
        return 0.0
    try:
        return float(str(value).strip().replace(".", "").replace(",", "."))
    except (ValueError, TypeError):
        return 0.0


def fmt_lit(value) -> str:
    try:
        return f"{float(value):,.0f} L" if pd.notna(value) and float(value) != 0 else "—"
    except (ValueError, TypeError):
        return "—"


def fmt_imp(value) -> str:
    try:
        return f"$ {float(value):,.0f}" if pd.notna(value) else "—"
    except (ValueError, TypeError):
        return "—"


def fmt_pct(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "—"
    try:
        f = float(value)
        arrow = "▲" if f >= 0 else "▼"
        return f"{arrow} {f:+.1f}%"
    except (ValueError, TypeError):
        return "—"


def variacion_pct(actual, referencia):
    if referencia in (0, None) or pd.isna(referencia):
        return None
    return (actual / referencia - 1) * 100


def map_linea(grupo_materiales_1: str) -> str:
    for categoria, nombres in LINEAS_PRINCIPALES.items():
        if grupo_materiales_1 in nombres:
            return categoria
    return "Otros"


def titulo(texto: str):
    print("\n" + "=" * 72)
    print(f"  {texto}")
    print("=" * 72)


def subtitulo(texto: str):
    print(f"\n  ── {texto} ──")


# ── Carga de datos ─────────────────────────────────────────────────────────────
print("Cargando datasets...")
df_forecast = pd.read_excel(DATASET / "forecast_fbda_limpio.xlsx")
df_cabecera = pd.read_excel(DATASET / "cabecera_ventas_fbda_limpio.xlsx")
df_posicion = pd.read_excel(DATASET / "posicion_ventas_fbda_limpio.xlsx")

print(f"  forecast:  {len(df_forecast):,} filas")
print(f"  cabecera:  {len(df_cabecera):,} filas")
print(f"  posición:  {len(df_posicion):,} filas")


# ── Normalización ──────────────────────────────────────────────────────────────
# Factura como string limpio (sin .0) en ambas tablas
df_cabecera["_factura"] = (
    df_cabecera["Factura"]
    .astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
)
df_posicion["_factura"] = (
    df_posicion["Factura"]
    .astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
)

# Fecha como string AAAAMMDD para filtrar con startswith (equivale a LIKE '202503%')
df_cabecera["_fecha_str"] = (
    df_cabecera["Fecha asociada al pedido"]
    .astype(str).str.replace(r"\D", "", regex=True).str[:8]
)

# Vendedor como string sin ceros ni .0 (equivale al cast ::text + lstrip en SQL)
df_cabecera["_vendedor"] = (
    df_cabecera["Vendedor"]
    .astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")
)

# Litros en posicion con formato argentino (equivale al REPLACE/TRIM/::NUMERIC)
df_posicion["_lts"] = df_posicion[
    "Cantidad entregada efectivamente en UMV"
].apply(parse_num_ar)

# Forecast: cantidad numérica y zona/linea como string
df_forecast["_cant"]  = pd.to_numeric(df_forecast["Cantidad"], errors="coerce").fillna(0.0)
df_forecast["_zona"]  = df_forecast["Zona de ventas"].astype(str).str.strip()
df_forecast["_linea"] = df_forecast["Grupo de materiales 1"].astype(str).str.strip()

print("Normalización OK")


# ── Funciones (equivalentes a los CTEs SQL) ────────────────────────────────────
def get_facturas(vendedor_id: str, anio: int, mes: int) -> list:
    """
    CTE facturas_validas:
      SELECT DISTINCT Factura FROM cabecera
      WHERE Vendedor = ? AND Fecha LIKE 'AAAAMM%' AND Tipo = 'M'
    """
    prefijo = f"{anio}{mes:02d}"
    mask = (
        (df_cabecera["_vendedor"] == vendedor_id) &
        (df_cabecera["_fecha_str"].str.startswith(prefijo)) &
        (df_cabecera["Tipo de documento comercial"] == "M")
    )
    return df_cabecera[mask]["_factura"].unique().tolist()


def get_real_mtd(vendedor_id: str, anio: int, mes: int,
                 linea: str = None, zona: str = None) -> float:
    """
    CTE real_mtd:
      SUM(lts) de posicion filtrado por facturas válidas del mes,
      opcionalmente por línea y zona.
    """
    facturas = get_facturas(vendedor_id, anio, mes)
    if not facturas:
        return 0.0

    mask = df_posicion["_factura"].isin(facturas)
    if linea:
        mask &= df_posicion["Grupo de materiales 1"] == linea
    if zona:
        mask &= df_posicion["Zona de ventas del pedido de cliente"].astype(str) == zona

    return float(df_posicion[mask]["_lts"].sum())


def get_forecast_mtd(zona: str, anio: int, mes: int, linea: str = None) -> float:
    """
    CTE fc_mtd:
      SUM(Cantidad) de forecast filtrado por zona, año, mes,
      opcionalmente por línea.
    """
    mask = (
        (df_forecast["_zona"] == zona) &
        (df_forecast["Año"] == anio) &
        (df_forecast["Mes"] == mes)
    )
    if linea:
        mask &= df_forecast["_linea"] == linea

    return float(df_forecast[mask]["_cant"].sum())


def kpi_mtd(vendedor_id: str, anio: int, mes: int,
            linea: str = None, zona: str = None) -> dict:
    """
    SELECT final que combina real_mtd y fc_mtd y calcula
    Variacion Absoluta, Variacion % y Cumplimiento %.
    Equivale al SELECT final del query SQL completo.
    """
    zona_fc = zona or VENDEDORES.get(vendedor_id, {}).get("zona", "")

    real = get_real_mtd(vendedor_id, anio, mes, linea=linea, zona=zona)
    fc   = get_forecast_mtd(zona_fc, anio, mes, linea=linea)

    var_abs = real - fc
    var_pct = variacion_pct(real, fc)
    cumpl   = round(real / fc * 100, 1) if fc else None

    return {
        "Real MTD (lts)":      round(real, 1),
        "Forecast MTD (lts)":  round(fc, 1),
        "Variacion Absoluta":  round(var_abs, 1),
        "Variacion %":         round(var_pct, 1) if var_pct is not None else None,
        "Cumplimiento %":      cumpl,
    }


# ── Ejemplo de uso ─────────────────────────────────────────────────────────────
# Equivale exactamente al query SQL con Vendedor=77, mes=3/2025, Sernova Vodka
resultado = kpi_mtd(
    vendedor_id="77",
    anio=2025,
    mes=3,
    linea="Sernova Vodka",
    zona="77",
)

titulo("MTD — Sernova Vodka | Vendedor 77 | Marzo 2025")
print(f"  Real MTD          : {fmt_lit(resultado['Real MTD (lts)'])}")
print(f"  Forecast MTD      : {fmt_lit(resultado['Forecast MTD (lts)'])}")
print(f"  Variación Absoluta: {fmt_lit(resultado['Variacion Absoluta'])}")
print(f"  Variación %       : {fmt_pct(resultado['Variacion %'])}")
print(f"  Cumplimiento      : {resultado['Cumplimiento %']:.1f}%"
      if resultado["Cumplimiento %"] else "  Cumplimiento      : —")

Cargando datasets...
  forecast:  42,600 filas
  cabecera:  4,106 filas
  posición:  215,871 filas
Normalización OK

  MTD — Sernova Vodka | Vendedor 77 | Marzo 2025
  Real MTD          : 6,338 L
  Forecast MTD      : 5,331 L
  Variación Absoluta: 1,007 L
  Variación %       : ▲ +18.9%
  Cumplimiento      : 118.9%
